In [2]:
%pip install rplidar

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rplidar: filename=rplidar-0.9.2-py3-none-any.whl size=5459 sha256=bb90dc27611a2c9f326929332b7ab1f3de8ac2206e1c82c0878bb35b0e93beee
  Stored in directory: c:\users\x513\appdata\local\pip\cache\wheels\d4\a8\6d\c65a0fb33dd4249aad45916ee45bc8e82489e22d8c616fe8fb
Successfully built rplidar
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from rplidar import RPLidar
print("Bibliothèque importée avec succès !")

Bibliothèque importée avec succès !


In [8]:
from serial.tools import list_ports

for port in list_ports.comports():
    print(port.device, port.description)

COM6 Silicon Labs CP210x USB to UART Bridge (COM6)


In [9]:
from rplidar import RPLidar

lidar = RPLidar('COM6')

for scan in lidar.iter_scans():
    print(scan)
    break

lidar.stop()
lidar.stop_motor()
lidar.disconnect()



RPLidarException: Descriptor length mismatch

In [10]:
import os
import sys

print("Python utilisé :", sys.executable)
print("Dossier actuel :", os.getcwd())
print("Fichiers présents :", os.listdir())

Python utilisé : c:\Users\X513\anaconda3\python.exe
Dossier actuel : c:\Users\X513\Documents\LIDAR
Fichiers présents : ['LiDARpy.ipynb']


In [18]:

import asyncio
import csv
import math
import threading
import time
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from queue import Queue, Empty

import numpy as np
import matplotlib
matplotlib.use("TkAgg")
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

from scanner import RPLidar

# ---------------- CONFIGURATION ----------------
PORT = "COM6"
BAUDRATE = 460800
MAX_RANGE_M = 5.0 #portee zoom; 
DANGER_M = 0.50
WARNING_M = 1.20
FRONT_ANGLE_DEG = 0.0
FRONT_SECTOR_DEG = 30.0
# ------------------------------------------------

measurements = Queue(maxsize=10000)
stop_event = threading.Event()


async def lidar_reader():
    """Lit les mesures du RPLIDAR C1."""
    lidar = RPLidar(PORT, BAUDRATE, timeout=0.2)

    try:
        scan_task = asyncio.create_task(
            lidar.simple_scan(make_return_dict=False)
        )

        while not stop_event.is_set():
            try:
                item = await asyncio.wait_for(
                    lidar.output_queue.get(), timeout=0.25
                )
            except asyncio.TimeoutError:
                continue

            try:
                angle = float(item["a_deg"]) % 360
                distance_mm = float(item["d_mm"])
                quality = int(item.get("q", 0))
            except (KeyError, TypeError, ValueError):
                continue

            if distance_mm <= 0:
                continue

            # Ne pas bloquer la lecture si l'affichage est en retard.
            if measurements.full():
                try:
                    measurements.get_nowait()
                except Empty:
                    pass

            measurements.put_nowait((angle, distance_mm, quality))

        if hasattr(lidar, "stop_event"):
            lidar.stop_event.set()

        try:
            await asyncio.wait_for(scan_task, timeout=2)
        except Exception:
            scan_task.cancel()
            await asyncio.gather(scan_task, return_exceptions=True)

    finally:
        if hasattr(lidar, "shutdown"):
            result = lidar.shutdown()
            if asyncio.iscoroutine(result):
                await result


def reader_thread():
    try:
        asyncio.run(lidar_reader())
    except Exception as exc:
        measurements.put(("ERROR", str(exc), 0))


def in_sector(angle, center, width):
    delta = (angle - center + 180) % 360 - 180
    return abs(delta) <= width


class LidarApp:
    def __init__(self, root):
        self.root = root
        self.root.title("RPLIDAR C1 | Radar temps réel")
        self.root.geometry("1050x720")
        self.points = {}

        top = ttk.Frame(root, padding=10)
        top.pack(fill="x")

        ttk.Label(
            top, text="RPLIDAR C1",
            font=("Segoe UI", 18, "bold")
        ).pack(side="left")

        self.status = tk.StringVar(value="Connexion en cours...")
        ttk.Label(top, textvariable=self.status).pack(side="right")

        main = ttk.Frame(root, padding=10)
        main.pack(fill="both", expand=True)

        plot_frame = ttk.Frame(main)
        plot_frame.pack(side="left", fill="both", expand=True)

        self.fig = Figure(figsize=(7, 6), tight_layout=True)
        self.ax = self.fig.add_subplot(111, projection="polar")
        self.ax.set_theta_zero_location("N")
        self.ax.set_theta_direction(-1)
        self.ax.set_ylim(0, MAX_RANGE_M)
        self.ax.set_title("Radar 360° : distances en mètres")
        self.ax.grid(True, alpha=0.4)

        angles = np.deg2rad(
            np.linspace(-FRONT_SECTOR_DEG, FRONT_SECTOR_DEG, 60)
        )
        self.ax.fill_between(
            angles, 0, MAX_RANGE_M, color="orange", alpha=0.12
        )

        self.scatter = self.ax.scatter([], [], s=12)

        self.canvas = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)

        panel = ttk.LabelFrame(main, text="Analyse", padding=12)
        panel.pack(side="right", fill="y", padx=(10, 0))

        self.minimum = tk.StringVar(value="--")
        self.front = tk.StringVar(value="--")
        self.count = tk.StringVar(value="0")
        self.alert = tk.StringVar(value="En attente de mesures")

        for label, var in [
            ("Distance minimale", self.minimum),
            ("Distance devant", self.front),
            ("Points reçus", self.count),
            ("Alerte", self.alert),
        ]:
            ttk.Label(panel, text=label).pack(anchor="w", pady=(8, 0))
            ttk.Label(
                panel, textvariable=var,
                font=("Segoe UI", 11, "bold"),
                wraplength=200
            ).pack(anchor="w")

        ttk.Button(
            panel, text="Exporter CSV", command=self.export_csv
        ).pack(fill="x", pady=(20, 6))

        ttk.Button(
            panel, text="Arrêter", command=self.close
        ).pack(fill="x")

        self.root.protocol("WM_DELETE_WINDOW", self.close)
        self.root.after(100, self.update_display)

    def update_display(self):
        if stop_event.is_set():
            return

        # Consomme toutes les nouvelles mesures disponibles.
        while True:
            try:
                item = measurements.get_nowait()
            except Empty:
                break

            if item[0] == "ERROR":
                self.status.set("Erreur : " + item[1])
                continue

            angle, distance_mm, quality = item
            distance_m = distance_mm / 1000.0

            if 0.05 <= distance_m <= MAX_RANGE_M:
                self.points[round(angle) % 360] = (
                    distance_m, quality, time.time()
                )

        # Retire les points qui n'ont pas été actualisés depuis 2 s.
        now = time.time()
        self.points = {
            a: p for a, p in self.points.items()
            if now - p[2] < 2.0
        }

        valid = [
            (a, p[0]) for a, p in self.points.items()
        ]

        if valid:
            theta = np.deg2rad([a for a, d in valid])
            radius = [d for a, d in valid]

            self.scatter.remove()
            self.scatter = self.ax.scatter(
                theta, radius, s=12
            )

            distances = [d for a, d in valid]
            self.minimum.set(f"{min(distances):.2f} m")

            front_distances = [
                d for a, d in valid
                if in_sector(a, FRONT_ANGLE_DEG, FRONT_SECTOR_DEG)
            ]

            if front_distances:
                front_min = min(front_distances)
                self.front.set(f"{front_min:.2f} m")

                if front_min < DANGER_M:
                    self.alert.set("DANGER : obstacle proche")
                elif front_min < WARNING_M:
                    self.alert.set("ATTENTION : obstacle détecté")
                else:
                    self.alert.set("Rien de proche détecté")
            else:
                self.front.set("--")
                self.alert.set("Aucune mesure devant")

            self.count.set(str(len(valid)))
            self.status.set("Réception des mesures")

            self.canvas.draw_idle()

        self.root.after(100, self.update_display)

    def export_csv(self):
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("Fichiers CSV", "*.csv")],
            initialfile="scan_lidar.csv"
        )
        if not filename:
            return

        with open(filename, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=";")
            writer.writerow(["angle_deg", "distance_m", "qualite"])
            for angle, (distance, quality, timestamp) in sorted(
                self.points.items()
            ):
                writer.writerow([angle, distance, quality])

        messagebox.showinfo("Export", "Mesures exportées.")

    def close(self):
        stop_event.set()
        self.root.after(300, self.root.destroy)


if __name__ == "__main__":
    threading.Thread(
        target=reader_thread, daemon=True
    ).start()

    root = tk.Tk()
    app = LidarApp(root)
    root.mainloop()


In waiting: 3
In waiting: 0
